In [ ]:
"""
即時剪貼簿去識別化 / 還原工具 (Microsoft Presidio)
- 複製含敏感資訊的文字     -> 自動遮蔽成 <ENTITY_n> 標籤並貼回剪貼簿
- 複製已含 <ENTITY_n> 標籤的文字 -> 不掃描，直接查表還原成原始敏感資訊並貼回剪貼簿
- 標籤對應存於 clipboard_map.txt (格式: 標籤\t原始內容)

安裝: pip install presidio-analyzer presidio-anonymizer pyperclip pywin32
      python -m spacy download en_core_web_lg
"""
import re
import time
import pyperclip
from dataclasses import dataclass
from typing import Callable, Optional
from presidio_analyzer import AnalyzerEngine

MAP_FILE = "clipboard_map.txt"
TAG_RE = re.compile(r"<([A-Z_]+)_(\d+)>")

analyzer = AnalyzerEngine()


def luhn_valid(value: str) -> bool:
    """信用卡卡號 Luhn 演算法驗證，供 RULES 的 validator 使用"""
    digits = [int(d) for d in value if d.isdigit()]
    if not (13 <= len(digits) <= 19):
        return False
    checksum, parity = 0, len(digits) % 2
    for i, d in enumerate(digits):
        if i % 2 == parity:
            d *= 2
            if d > 9:
                d -= 9
        checksum += d
    return checksum % 10 == 0


# ========== 正則快速比對規則區：想增加/刪除規則，只要在下面 RULES 這個 list 新增/刪除一行 Rule(...) ==========
@dataclass
class Rule:
    name: str                                          # 標籤用的類別名稱 (etype)
    pattern: str                                        # 正則表達式字串
    flags: int = 0                                      # re flags，例如 re.IGNORECASE
    validator: Optional[Callable[[str], bool]] = None   # 二次驗證函式（可選，降低誤報）

RULES = [
    Rule("JWT", r"\beyJ[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\b"),
    Rule("API_KEY_OPENAI", r"\bsk-(?:proj-|ant-)?[A-Za-z0-9_-]{20,}\b"),   # 支援新版 sk-proj-... 格式
    Rule("API_KEY_AWS", r"\b(?:AKIA|ASIA)[0-9A-Z]{16}\b"),
    Rule("API_KEY_GITHUB", r"\b(?:ghp|gho|ghu|ghs|ghr|github_pat)_[A-Za-z0-9_]{20,}\b"),
    Rule("API_KEY_GOOGLE", r"\bAIza[0-9A-Za-z\-_]{35}\b"),
    Rule("TW_PHONE", r"\b09\d{2}[- ]?\d{3}[- ]?\d{3}\b"),
    Rule("CREDIT_CARD", r"\b(?:\d[ -]?){13,19}\b", validator=luhn_valid),
    # 通用長亂碼字串（如 Unsplash access/secret key）：長度>=30、須同時含英文字母與數字，避免誤判一般文字
    Rule("GENERIC_SECRET", r"\b(?=[A-Za-z0-9_-]{30,}\b)(?=[A-Za-z0-9_-]*[A-Za-z])(?=[A-Za-z0-9_-]*[0-9])[A-Za-z0-9_-]{30,}\b"),
    # 範例：新增自己的規則就照這個格式加一行，例如：
    # Rule("TW_ID", r"\b[A-Z][12]\d{8}\b", validator=tw_id_valid),
]

_COMPILED_RULES = [(r.name, re.compile(r.pattern, r.flags), r.validator) for r in RULES]


def regex_filter(text):
    """快速正則比對，回傳格式與 presidio_filter 相同 [(start,end,entity_type),...]"""
    matches = []
    for name, pattern, validator in _COMPILED_RULES:
        for m in pattern.finditer(text):
            if validator is None or validator(m.group(0)):
                matches.append((m.start(), m.end(), name))
    return matches


# ---------- 過濾器區：想加/換偵測方式，寫一個回傳 [(start,end,entity_type),...] 的函式，append 進 FILTERS 即可 ----------
def presidio_filter(text):
    return [(r.start, r.end, r.entity_type) for r in analyzer.analyze(text=text, language="en")]


FILTERS = [regex_filter, presidio_filter]   # 先跑快速正則比對，再進 Presidio 模型掃描


# ---------- 對應表讀寫 ----------
def load_map():
    mapping = {}
    try:
        with open(MAP_FILE, encoding="utf-8") as f:
            for line in f:
                tag, _, val = line.rstrip("\n").partition("\t")
                if tag:
                    mapping[tag] = val
    except FileNotFoundError:
        pass
    return mapping


def save_map(mapping):
    with open(MAP_FILE, "w", encoding="utf-8") as f:
        for tag, val in mapping.items():
            f.write(f"{tag}\t{val}\n")


def merge_overlaps(matches):
    """多個過濾器/規則可能比對到重疊區間，若直接替換會切壞字串，
    這裡依起始位置排序、同起點取較長者，彼此重疊的區間只保留先出現(較長)的那個"""
    matches = sorted(matches, key=lambda m: (m[0], -(m[1] - m[0])))
    merged, last_end = [], -1
    for start, end, etype in matches:
        if start >= last_end:
            merged.append((start, end, etype))
            last_end = end
    return merged


# ---------- 遮蔽 / 還原 ----------
def anonymize(text, mapping):
    matches = []
    for flt in FILTERS:
        matches += flt(text)
    if not matches:
        return None
    matches = merge_overlaps(matches)

    reverse = {v: k for k, v in mapping.items()}
    counters = {}
    for tag in mapping:
        t, i = TAG_RE.match(tag).groups()
        counters[t] = max(counters.get(t, 0), int(i))

    out = text
    for start, end, etype in sorted(matches, key=lambda m: m[0], reverse=True):
        val = text[start:end]
        tag = reverse.get(val)
        if not tag:
            counters[etype] = counters.get(etype, 0) + 1
            tag = f"<{etype}_{counters[etype]}>"
            mapping[tag] = val
            reverse[val] = tag
        out = out[:start] + tag + out[end:]

    save_map(mapping)
    return out


def deanonymize(text, mapping):
    return TAG_RE.sub(lambda m: mapping.get(m.group(0), m.group(0)), text)


# ---------- 主迴圈 ----------
def main():
    mapping = load_map()
    last = ""
    print("監控剪貼簿中... (Ctrl+C 結束)")
    while True:
        try:
            text = pyperclip.paste()
        except Exception:
            text = ""

        if text and text != last:
            new_text = deanonymize(text, mapping) if TAG_RE.search(text) else anonymize(text, mapping)
            if new_text and new_text != text:
                pyperclip.copy(new_text)
                text = new_text
            last = text

        time.sleep(0.5)


if __name__ == "__main__":
    main()